---
title: "DRG Cleaning v2"

author: "Carlos Resurreccion"

date: "2024-10-21"

---


# Parameters

Change which year to process in
`~/drg-pipeline/data-cleaning/cache/year_to_load.txt`

Change other rarely touched parameters in
`~/drg-pipeline/data-cleaning/r_scripts_v2/00_v2_params-fpaths.R`


In [ ]:
# Delete all R objects and run garbage collection so we start with a clean slate
rm(list = ls())
invisible(gc())

thread_offset <- 0

sample_size_divisor <- 625

# Whether to sample each split_part by sample_size_divisor
# (useful when iterating through code runs in quick succession)
to_sample <- FALSE
# TODO: Add description here
to_write <- TRUE
# TODO: Add description here
to_flush <- FALSE
# TODO: Add description here
to_parallel <- TRUE
cat("Parallelization:", to_parallel, "\n")
# TODO: Add description here
to_debug <- FALSE
verbose_output <- if (to_debug) TRUE else FALSE

to_python <- FALSE
to_thai <- TRUE
to_spc <- FALSE

to_generate_thai <- FALSE

to_thai_all_years <- FALSE

to_bq <- TRUES

to_generate_subset <- FALSE


Parallelization: TRUE 


# Libraries


In [2]:
# Update the grouper
system("git submodule update --init --recursive")

# List, install (if applicable), and load packages
## Required packages
required_packages <- c(
  "data.table", "here", "tictoc", "stringr", "stringi", "lubridate",
  "profvis", "hash", "future", "future.apply", "knitr", "htmlwidgets",
  "parallelly", "stringdist", "parallel", "reticulate", "bigrquery",
  "jsonlite", "googleCloudStorageR", "haven", "fst", "httr", "ggplot2",
  "rmarkdown"
  # , "docstring", "progress" # Comma is here so if I uncomment this line it
  # automatically works without having to type or delete a comma after haven
)

# Additional packages to install via remotes (GitHub), if not available
github_packages <- c("r-lib/styler")

# Function to install and load packages quietly
install_and_load <- function(package) {
  if (!require(package, character.only = TRUE)) {
    message("Installing ", package)
    install.packages(package, dependencies = TRUE)
  } else {
    if (verbose_output) message("Loading ", package)
  }
  library(package, character.only = TRUE)
}

# Function to install packages from GitHub via remotes
install_from_github <- function(repo) {
  package_name <- basename(repo)
  if (!require(package_name, character.only = TRUE)) {
    if (!require("remotes", character.only = TRUE)) {
      install.packages("remotes")
    }
    message("Installing ", package_name, " from GitHub (", repo, ")")
    remotes::install_github(repo)
  } else {
    if (verbose_output) message("Loading ", package_name)
  }
  library(package_name, character.only = TRUE)
}

# Apply the function to each required package
message("Installing/loading required CRAN packages...")
invisible(
  suppressPackageStartupMessages(
    lapply(required_packages, install_and_load)
  )
)

# Install and load GitHub packages if not installed
message("Installing/loading required GitHub packages...")
invisible(
  suppressPackageStartupMessages(
    lapply(github_packages, install_from_github)
  )
)

# detect available threads
nthreads <- parallelly::availableCores()


Installing/loading required CRAN packages...

Installing/loading required GitHub packages...



# R Scripts


In [3]:
scripts_path <- here("data-cleaning/r_scripts_v2")

# List all R files in the directory with full paths, sorted by filename
r_files <- list.files(scripts_path, pattern = "\\.R$", full.names = TRUE)

# Source each file sequentially
for (file in r_files) {
  if (verbose_output) message(Sys.time(), " Sourcing: ", file)
  invisible(source(file))
}

message(year_to_load)

bq_dataset <- "drg_claims"


All directories exist.


Total Rows via cached object: 11777674

Utilizing 4 cores (8 threads)




Using virtual environment 'r-reticulate' ...


+ /home/resurreccion_cmc_gmail_com/.virtualenvs/r-reticulate/bin/python -m pip install --upgrade --no-user pip

2018



# Load Full Claims from GCS


In [4]:
# Load raw claims from GCS only if they don't exist on the VM yet
for (year in 2018:2023) {
  # Assign the correct file extension based on the year
  file_type <- if (year %in% c(2022:2023)) ".tsv" else ".csv"
  file_name <- paste0(full_claims_prefix, year, file_type)
  bq_name <- paste0(full_claims_bq_prefix, year, file_type)

  # Check if the file exists in the target directory
  file_path <- here(raw_claims_path, file_name)
  exists <- file.exists(file_path)

  # If the file does not exist, run the gsutil cp command
  if (!exists) {
    if (!is.null(gcp_proj) && gcp_proj == "drg-pipeline") {
      system(
        paste0(
          "cd .. && gsutil cp gs://phic-claims-raw/",
          bq_name, " ", raw_claims_path
        ),
        intern = FALSE, ignore.stderr = FALSE
      )
    } else {
      stop("Error: GCP Project is not null and is not drg-pipeline")
    }
  } else {
    next
    # message(paste(
    #   "File", file_name,
    #   "already exists in the target directory. Skipping download.\n"
    # ))
  }
}


# Load Mapping Data


In [5]:
# Enable caching and printing options for data mapping
to_use_cache <- TRUE # Set to TRUE to enable saving and loading of .rds files
to_print_mapping_data <- TRUE # Set to TRUE to print mapping data tables

# Helper function to load data from cache or query from BigQuery if not cached
load_or_query <- function(query, var_name) {
  rds_path <- here(cache_path, "mapping", paste0(var_name, ".rds"))
  if (to_use_cache && file.exists(rds_path)) {
    if (verbose_output) message("Loading ", var_name, " from cache...")
    # Load data from .rds file if cache exists
    return(readRDS(rds_path))
  } else {
    if (verbose_output) message("Querying ", var_name, " from BigQuery...")
    # Query data from BigQuery if not cached
    # Query execution function (BigQuery to data.table)
    dt <- query_bq_to_dt(query)
    saveRDS(dt, rds_path) # Save queried data to .rds cache file
    return(dt)
  }
}

# Helper function to print all rows of a data.table if
# to_print_mapping_data is enabled
if (to_print_mapping_data) {
  print_all <- function(dt, title) {
    cat("\n---", title, "---\n") # Print table title
    print(dt, nrow = Inf) # Print all rows of the data.table
  }
}

# 1. Query and load the `grouper_v5.proc` table
# This table contains procedure codes and attributes
# like description, classification, and site
proc_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.proc`")
proc <- load_or_query(proc_query, "proc")
proc[, CODE := as.character(CODE)] # Ensure the CODE column is of character type

# 2. Query and load the `phic.acr_rvs_map` table
# This table maps RVS codes to ICD-9-CM codes,
# used for healthcare billing purposes
rvs_icd9_query <- paste0("SELECT * FROM `", gcp_proj, ".phic.acr_rvs_map`")
rvs_icd9 <- load_or_query(rvs_icd9_query, "rvs_icd9")

# Convert RVS and ICD9CM columns to character type
# and adjust ICD9CM for multiplication
rvs_icd9 <- rvs_icd9[, .(
  rvs = as.character(rvs),
  icd9cm = as.character(as.numeric(icd9cm) * 100)
)]

# Merge the RVS-ICD9 mapping with the proc table for DRG classification
rvs_icd9 <- merge(
  rvs_icd9,
  proc[, .(CODE, DRGUSE)], # Select CODE and DRGUSE columns for merging
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
  # Merge on icd9cm and CODE columns
)

# Filter and annotate DRG-related codes, removing unnecessary DRGUSE column
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE][
  !is.na(rvs) & !is.na(icd9cm), -"DRGUSE"
]

# 3. Query and load `phic.acr_procedure` table
# This table contains RVS codes, relative value units (RVUs),
# and descriptions for procedures
acr_rvs_query <- paste0("SELECT * FROM `", gcp_proj, ".phic.acr_procedure`")
acr_rvs <- load_or_query(acr_rvs_query, "acr_rvs")

# 4. Query and load `grouper_v5.i10` table
# This table contains ICD-10 codes with DRG grouping data,
# including codes marked as "accepted" (ACCPDX = "Y")
i10_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10`")
tdrg_icd10 <- load_or_query(i10_query, "tdrg_icd10")
setkey(tdrg_icd10, "CODE") # Set the CODE column as key for efficient lookups

# Extract unique accepted ICD-10 codes for diagnosis
# (ACCPDX == "Y") and store in acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])

# Create an environment for quick lookup of accepted diagnosis codes
acc_pdx_env <- new.env(hash = TRUE, parent = emptyenv())
for (code in acc_pdx) {
  # Assign each accepted code to the environment
  assign(code, TRUE, envir = acc_pdx_env)
}

# 5. Query and load `icd.phl_icd10` table
# This table lists diseases and their corresponding
# ICD-10 codes specific to the Philippines
phl_icd10_query <- paste0("SELECT * FROM `", gcp_proj, ".icd.phl_icd10`")
phl_icd10 <- load_or_query(phl_icd10_query, "phl_icd10")

# Filter and process neoplasm codes by extracting
# specific codes from complex ICD-10 notations
neoplasms_dt_actual <- as.data.table(phl_icd10[
  # Select rows with '/' in icd10, indicating neoplasm codes
  grepl("/", icd10), .(icd10)
  # Extract relevant part
][, icd10 := sapply(strsplit(icd10, ","), function(x) trimws(x[2]))])


# 6. Query and load `grouper_v5.i10vx` table
# This table contains an expanded version of ICD-10 codes with validation flags
i10vx_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10vx`")
i10vx <- load_or_query(i10vx_query, "i10vx")
setkey(i10vx, "code") # Set the code column as key for efficient lookup
acc_icd <- unique(i10vx[, code]) # Extract unique ICD codes from this table
acc_icd_set <- unique(acc_icd)

# 7. Query and load `hci.temp_hci` table
# This table lists healthcare institutions with details
# like ownership, category, and location
hci_query <- paste0("SELECT * FROM `", gcp_proj, ".hci.temp_hci`")
hci <- load_or_query(hci_query, "hci")

# 8. Define global variables for use later in the script:
neoplasm_codes <- unique(neoplasms_dt_actual$icd10) # Unique neoplasm codes
covid_codes <- unique(covid_rvs) # Unique COVID-related codes
rvs_codes <- unique(acr_rvs$rvs) # Unique RVS codes

neoplasm_pattern <- paste0("(", paste(neoplasm_codes, collapse = "|"), ")")
covid_pattern <- paste0("(", paste(covid_codes, collapse = "|"), ")")
rvs_pattern <- paste0("(", paste(rvs_codes, collapse = "|"), ")")

phil_icds <- unique(gsub("[^A-Za-z0-9]", "", phl_icd10[!grepl("/", icd10), icd10]))
icd_codes <- unique(tdrg_icd10$CODE)

# Function to create an environment from a vector of unique values
create_env_from_vector <- function(vec) {
  env <- new.env(parent = emptyenv())
  list2env(setNames(as.list(rep(TRUE, length(vec))), vec), envir = env)
  return(env)
}

# 1. Create environment for `proc` table data if specific values are needed
# Here we assume `proc$CODE` is the field of interest
proc_env <- create_env_from_vector(proc$CODE)

# 2. Create environment for `rvs_icd9` table data based on `rvs` and `icd9cm`
rvs_env <- create_env_from_vector(rvs_icd9$rvs)
icd9cm_env <- create_env_from_vector(rvs_icd9$icd9cm)

# 3. Environment for `acr_rvs` table (assuming `rvs` is the field of interest)
acr_rvs_env <- create_env_from_vector(acr_rvs$rvs)

# 4. Environment for accepted ICD-10 codes (from `tdrg_icd10`)
acc_pdx_env <- create_env_from_vector(acc_pdx)

# 5. Environment for `phl_icd10` ICD-10 codes (e.g., neoplasm codes)
phl_icd10_env <- create_env_from_vector(phl_icd10$icd10)

# 6. Environment for expanded ICD-10 codes (`i10vx`)
acc_icd_env <- create_env_from_vector(i10vx$code)

# 7. Environment for `hci` table data if needed for specific fields (e.g., `id_hci`)
# Assuming `hci$id_hci` is the identifier of interest
hci_env <- create_env_from_vector(hci$id_hci)

# 8. Other specific environments for global variables
neoplasm_env <- create_env_from_vector(neoplasm_codes)
covid_env <- create_env_from_vector(covid_codes)
rvs_codes_env <- create_env_from_vector(rvs_codes)
phil_icds_env <- create_env_from_vector(phil_icds)
icd_codes_env <- create_env_from_vector(icd_codes)

# Combine COVID and neoplasm codes into a single environment
covid_neoplasm_codes <- unique(c(covid_codes, neoplasm_codes))
covid_neoplasm_env <- create_env_from_vector(covid_neoplasm_codes)

# Combine COVID, RVS, and neoplasm codes into a single environment for efficient lookup
covid_rvs_neoplasm_codes <- unique(c(covid_codes, rvs_codes, neoplasm_codes))
covid_rvs_neoplasm_env <- create_env_from_vector(covid_rvs_neoplasm_codes)


# Create a combined regular expression pattern to match COVID,
# RVS, and neoplasm codes in data processing
covid_rvs_neoplasm_pattern <- paste(
  c(covid_codes, rvs_codes, neoplasm_codes),
  collapse = "|"
)
# if (to_print_mapping_data) print(covid_rvs_neoplasm_pattern)
# # Print regex pattern if enabled

# Helper function to save all specified data tables into a
# single text file for debugging
save_all_data_to_file <- function(file_path, ...) {
  args <- list(...)
  sink(file_path) # Redirect output to the specified file
  cat("\n--- All Data Tables in One View ---\n") # Header for the file
  for (name in names(args)) {
    cat("\n---", name, "---\n") # Print table name as a header within the file
    # Print all rows of each data.table
    print(args[[name]], nrow = Inf, max.print = Inf)
  }
  sink() # Stop redirecting output to the file
  if (verbose_output) message("All data tables saved to ", file_path) # Confirmation message
}

# Set the file path for the output text file, where all
# data tables will be saved
output_file <- here(debug_path, "mapping_data.txt")

# If enabled, save all processed data tables to a single specified
# file for debugging and verification.
# Each table represents a different aspect of the medical coding,
# classification, and healthcare provider data.
if (to_print_mapping_data) {
  options(max.print = 999999)
  save_all_data_to_file(
    output_file,

    # Data table containing procedure codes and attributes
    # for each procedure code.
    # Columns include CODE (unique procedure identifier),
    # DRGUSE (flag indicating if the procedure is used for DRG grouping),
    # and several other attributes related to procedure
    # classification, gender applicability, site, and level of care.
    grouper_v5_proc = proc,

    # Mapping between ICD-9-CM codes and RVS (Relative Value Scale)
    # codes used in medical billing.
    # Includes `is_drg` column to mark codes used in DRG grouping
    # after merging with the `grouper_v5.proc` table.
    # This table links standard ICD-9 procedure codes to specific
    # RVS codes for billing purposes.
    phic_acr_rvs_map = rvs_icd9,

    # Table of RVS codes and their associated RVU (Relative Value Units)
    # which represent the value of a procedure.
    # Also includes a detailed description of each procedure, such as type,
    #  category, or specific details about the procedure.
    # This table is essential for understanding the cost/value of each
    # RVS-coded procedure in medical billing.
    phic_acr_procedure = acr_rvs,

    # ICD-10 table with additional classification details relevant for
    #  DRG (Diagnosis Related Group) mapping.
    # Columns include CODE (ICD-10 diagnosis code), ACCPDX
    # (accepted primary diagnosis flag), and other grouping
    # variables like MDC (Major Diagnostic Category) and CC
    # (Complication/Comorbidity), which help classify the severity or
    # complexity of cases for healthcare reimbursement.
    grouper_v5_i10 = tdrg_icd10,

    # A unique list of ICD-10 codes flagged as ACCPDX (accepted
    # primary diagnosis codes) for use in DRG classification.
    # This list is derived from `grouper_v5.i10` and is used as
    # a quick reference to check if a diagnosis is eligible as a primary code.
    acc_pdx = acc_pdx,

    # Data specific to the Philippines for ICD-10 codes, containing
    # disease names and the corresponding ICD-10 codes.
    # This table includes the field `remarks`, which provides
    # special notes or guidance for each code, such as diagnostic
    # instructions or clarifications. This dataset is used to
    # manage and classify diseases in line with local health regulations.
    icd_phl_icd10 = phl_icd10,

    # Processed subset of neoplasm codes extracted from
    # `icd.phl_icd10`.
    # Contains ICD-10 codes specifically formatted to represent
    # malignant, benign, and other tumor types.
    # Useful for oncology-specific mappings in DRG processing
    # or cancer-related case management.
    neoplasms_dt_actual = neoplasms_dt_actual,

    # Expanded ICD-10 dataset with validation flags indicating
    # whether each code is valid.
    # Includes columns such as `validcode` (flag for validation status)
    # and `todel` (marker for codes that may need removal).
    # This dataset helps verify the validity of ICD-10 codes and
    #  manage code deprecation or updates.
    grouper_v5_i10vx = i10vx,

    # List of unique ICD-10 codes extracted from `grouper_v5.i10vx`
    # for quick access.
    # Acts as a condensed reference of all validated ICD-10 codes
    # available in the `grouper_v5.i10vx` dataset.
    # Useful for ensuring consistency and accuracy in ICD-10 code
    # usage across processes.
    acc_icd = acc_icd,

    # A directory of healthcare institutions (HCI), containing
    # detailed information about each provider,
    # including their institution name, ownership type (e.g.,
    # government, private), category, geographical details,
    # and provider classification. This table enables linkage
    # between clinical data and provider-specific data,
    # allowing for enhanced reporting and analytics on healthcare
    # service providers.
    hci_temp_hci = hci
  )
  # print(rvs_pattern)
  options(max.print = 1000)
}


In [6]:
if (to_generate_subset && (to_python || (to_thai && to_generate_thai) || to_spc)) {
  if (!to_thai_all_years) {
    # Load the data
    if (to_spc) {
      cat("\rReading stata\n")
      flush.console()
      result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata", ".rds")))
    } else {
      cat("\rReading final\n")
      flush.console()
      result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_time", ".rds")))
    }
    cat("\rComputing pat_bdate\n")
    result[, pat_bdate := NA_Date_]
    result[is.na(pat_bdate) & !is.na(pat_age), pat_bdate := as.Date(date_adm) - round(pat_age * 365.25)]
    cat("\rFiltering outpatient claims and outpatient NA\n")
    result <- result[clin_outpatient == FALSE]
    result[, caseid := as.character(seq_len(nrow(result)))]
    print(nrow(result))

    if (to_spc) {
      cat("\rWriting stata\n")
      flush.console()
      saveRDS(result, here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata_subset_with_bdate", ".rds")))
    } else {
      cat("\rWriting final\n")
      flush.console()
      saveRDS(result, here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_bdate_with_time", ".rds")))
    }
  } else if (to_thai_all_years) {
    for (year_to_load in c(2018:2023)) {
      year_to_load <<- year_to_load
      year_to_load <- year_to_load
      # Load the data
      if (to_spc) {
        cat("\rReading stata\n")
        flush.console()
        result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata", ".rds")))
      } else {
        cat("\rReading final\n")
        flush.console()
        result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_time", ".rds")))
      }
      cat("\rComputing pat_bdate\n")
      result[, pat_bdate := NA_Date_]
      result[is.na(pat_bdate) & !is.na(pat_age), pat_bdate := as.Date(date_adm) - round(pat_age * 365.25)]
      cat("\rFiltering outpatient claims and outpatient NA\n")
      result <- result[clin_outpatient == FALSE]
      result[, caseid := as.character(seq_len(nrow(result)))]
      print(nrow(result))

      if (to_spc) {
        cat("\rWriting stata\n")
        flush.console()
        saveRDS(result, here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata_subset_with_bdate", ".rds")))
      } else {
        cat("\rWriting final\n")
        flush.console()
        saveRDS(result, here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_bdate_with_time", ".rds")))
      }
    }
  }
  # Check for duplicates in id_series
  if (!to_thai_all_years && any(duplicated(result$id_series))) {
    # Identify duplicates
    duplicate_ids <- result$id_series[duplicated(result$id_series)]

    # Extract rows with duplicate id_series
    duplicate_rows <- result[id_series %in% duplicate_ids, ]

    # Print rows with duplicates
    cat("Rows with duplicate 'id_series':\n")
    print(duplicate_rows)

    # Stop execution
    stop("The 'id_series' column contains duplicates. Execution stopped.")
  }
  print(nrow(result))
  # Check for duplicates in id_series
  if (!to_thai_all_years && any(duplicated(result$caseid))) {
    # Identify duplicates
    duplicate_ids <- result$caseid[duplicated(result$caseid)]

    # Extract rows with duplicate id_series
    duplicate_rows <- result[caseid %in% duplicate_ids, ]

    # Print rows with duplicates
    cat("Rows with duplicate 'caseid':\n")
    print(duplicate_rows)

    # Stop execution
    stop("The 'caseid' column contains duplicates. Execution stopped.")
  }
  print(nrow(result))
}


Reading final
Computing pat_bdate
Filtering outpatient claims and outpatient NA
[1] 8338176
Writing final
Reading final
Computing pat_bdate
Filtering outpatient claims and outpatient NA
[1] 8679490
Writing final
Reading final
Computing pat_bdate
Filtering outpatient claims and outpatient NA
[1] 5964646
Writing final
Reading final
Computing pat_bdate
Filtering outpatient claims and outpatient NA
[1] 4834629
Writing final
Reading final
Computing pat_bdate
Filtering outpatient claims and outpatient NA
[1] 5784271
Writing final
Reading final
Computing pat_bdate
Filtering outpatient claims and outpatient NA
[1] 6790706
Writing final
[1] 6790706
[1] 6790706


# Python

In [7]:
if (to_python && !to_thai_all_years && to_bq && !to_generate_subset) {
  if (to_spc) {
    cat("\rReading stata\n")
    flush.console()
    result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata_subset_with_bdate", ".rds")))
  } else {
    cat("\rReading final\n")
    flush.console()
    result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_bdate_with_time", ".rds")))
  }
}

if (to_python) {
  message("Renaming columns")
  # Convert relevant data types
  result[, patage := as.numeric(pat_age)]
  result[, patsex := as.character(pat_sex)]
  result[, birthweight := as.numeric(pat_bwt)]
  result[, discharge := as.integer(clin_discharge)]
  result[, ageday := as.integer(pat_ageday)]
  result[, pdx := clin_pdx]

  # Handle ICD splitting for columns that are lists of character vectors
  split_codes_from_list <- function(dt, column, prefix, max_cols) {
    split_list <- dt[[column]] # Extract the list column
    # Pad each list to the specified max_cols with NAs if not enough elements
    split_cols <- lapply(1:max_cols, function(i) sapply(split_list, function(x) if (length(x) >= i) x[[i]] else NA_character_))
    split_dt <- as.data.table(split_cols)
    setnames(split_dt, paste0(prefix, 1:max_cols))
    return(split_dt)
  }

  # Apply the function to split clin_sdx and clin_proc
  message("Splitting clin_sdx")
  sdx_columns <- split_codes_from_list(result, "clin_sdx", "sdx", 12)
  message("Splitting clin_proc")
  proc_columns <- split_codes_from_list(result, "clin_proc", "proc", 20)

  # Combine the split columns back into the result
  message("cbind results")
  result <- cbind(result, sdx_columns, proc_columns)

  # Replace NA in non-date columns with "None"
  # non_date_columns <- c("patsex", "pdx", paste0("sdx", 1:12), paste0("proc", 1:20))
  # result[, (non_date_columns) := lapply(.SD, function(x) ifelse(is.na(x), "None", x)), .SDcols = non_date_columns]

  # Prepare the final data table for writing
  for_fwrite <- result[, c(
    "id_series", "date_adm", "date_dis", "time_adm", "time_dis", "patage", "patsex", "discharge", "pdx",
    paste0("sdx", 1:12), paste0("proc", 1:20), "birthweight", "ageday"
  ), with = FALSE]
  message("Formatting date_adm")
  for_fwrite[, date_adm := format(date_adm, "%Y-%m-%d %H:%M:%S")]
  message("Formatting date_dis")
  for_fwrite[, date_dis := format(date_dis, "%Y-%m-%d %H:%M:%S")]

  for_fwrite[, time_adm := NULL]
  for_fwrite[, time_dis := NULL]

  # Write the final table to a CSV file
  message("Writing to csv")
  fwrite(for_fwrite, here(checkpoint_7_path, paste0(checkpoint_7b_prefix, suffix, ".csv")))
  message("Creating summary table")
  # Create a summary table that shows the count of non-null values for each column
  summary_table <- for_fwrite[, lapply(.SD, function(x) sum(!is.na(x))), .SDcols = names(for_fwrite)]

  # Transpose the summary table to make it more readable
  summary_table <- transpose(summary_table)
  setnames(summary_table, "Non-Null Count")
  summary_table[, Column := names(for_fwrite)]

  # Reorder the summary table to show the columns
  setcolorder(summary_table, c("Column", "Non-Null Count"))

  # Print the summary table
  message("Printing summary table")
  print(summary_table)
}


In [8]:
if (to_python) {
  pandas <- import("pandas")
  py$pandas_df <- pandas$DataFrame(as.data.frame(for_fwrite))

  py_run_string("
import pandas as pd
import numpy as np
from io import StringIO
import sys

# Capture the output in a string buffer
old_stdout = sys.stdout
sys.stdout = mystdout = StringIO()

# Convert the column types explicitly
pandas_df['patage'] = pd.to_numeric(pandas_df['patage'], errors='coerce')
pandas_df['birthweight'] = pd.to_numeric(pandas_df['birthweight'], errors='coerce')
pandas_df['discharge'] = pandas_df['discharge'].astype('Int64')

# Convert string columns to 'string' dtype and replace NA values with None
string_columns = ['id_series', 'patsex', 'pdx', 'sdx1', 'sdx2', 'sdx3', 'sdx4', 'sdx5', 'sdx6', 'sdx7', 'sdx8', 'sdx9', 'sdx10', 'sdx11', 'sdx12',
                  'proc1', 'proc2', 'proc3', 'proc4', 'proc5', 'proc6', 'proc7', 'proc8', 'proc9', 'proc10', 'proc11', 'proc12',
                  'proc13', 'proc14', 'proc15', 'proc16', 'proc17', 'proc18', 'proc19', 'proc20', 'date_adm', 'date_dis']

# Replace NA, pd.NA, '<NA>', 'None', 'NA' in string columns with None
pandas_df[string_columns] = pandas_df[string_columns].replace([pd.NA, np.nan, '<NA>', 'None', 'NA'], 'None')

pandas_df = pandas_df.replace(pd.NA, None)
pandas_df = pandas_df.replace(np.nan, None)
pandas_df = pandas_df.replace('<NA>', None)
pandas_df = pandas_df.replace('None', None)
pandas_df = pandas_df.replace('NA', None)
pandas_df = pandas_df.replace(-2147483648, None)

# Convert columns to string dtype after replacing the values
pandas_df[string_columns] = pandas_df[string_columns].astype('string')

# Replace -2147483648 with None in numeric columns, if necessary TODO: why do we do this a second time?
pandas_df = pandas_df.replace(-2147483648, None)

# Filter rows where 'discharge' is not in [1, 2, 3, 4, 9]
not_in_list_values = pandas_df.loc[~pandas_df['discharge'].isin([1, 2, 3, 4, 9]), 'discharge']

# Get unique values and their counts
unique_not_in_list_values = not_in_list_values.value_counts()

# Print the unique values and their counts
print(unique_not_in_list_values)

# Reset stdout and capture the output
sys.stdout = old_stdout
output = mystdout.getvalue()

# Capture pandas_df.info() output
buffer = StringIO()
pandas_df.info(buf=buffer)
info_output = buffer.getvalue()

# Sample 10,000 rows from the DataFrame
# pandas_df = pandas_df.sample(n=10000, random_state=42)
")

  # Print the captured output in R
  cat(py$info_output)
}


In [9]:
if (to_python) {
  # Recalculate summary_table, considering 'None' as null
  summary_table <- for_fwrite[, lapply(.SD, function(x) sum(!is.na(x))), .SDcols = names(for_fwrite)]
  summary_table <- transpose(summary_table)
  setnames(summary_table, "NonNullCount_R")
  summary_table[, Column := names(for_fwrite)]
  setcolorder(summary_table, c("Column", "NonNullCount_R"))

  # Parse info_output from Python
  lines <- strsplit(py$info_output, "\n")[[1]]

  # Find the indices of the data lines
  start_idx <- which(grepl("^---", lines))
  end_idx <- which(grepl("^dtypes:", lines)) - 1

  data_lines <- lines[(start_idx + 1):end_idx]
  data_lines <- data_lines[nchar(data_lines) > 0]

  # Parse each line to extract the column name and non-null count
  parsed_lines <- str_match(data_lines, "^\\s*(\\d+)\\s+(\\S+)\\s+(\\d+)\\s+non-null\\s+(\\S+)")

  # Create python_summary data frame
  python_summary <- data.table(
    Column = parsed_lines[, 3],
    NonNullCount_Python = as.integer(parsed_lines[, 4])
  )

  # Merge the two summaries
  comparison <- merge(summary_table, python_summary, by = "Column", all = TRUE)

  # Calculate the difference
  comparison[, Difference := NonNullCount_R - NonNullCount_Python]

  # # Print the comparison
  # print(comparison)

  # Identify columns with differences
  differences <- comparison[Difference != 0]
  if (nrow(differences) == 0) {
    cat("All non-null counts match between R and Python.\n")
  } else {
    cat("Differences found in the following columns:\n")
    print(differences)
  }
}


In [10]:
if (to_python && !to_thai_all_years) {
  # Set pandas to display all rows and columns where 'patage' is NaN, then print everything using StringIO
  py_run_string("
import sys
from io import StringIO
import pandas as pd

# Set pandas display options to show all columns
pd.set_option('display.max_columns', None)

# Capture print output using StringIO
output = StringIO()
sys.stdout = output

# Filter rows where 'patage' is NaN and print the filtered DataFrame
filtered_df = pandas_df[pandas_df['patage'].isna()]
print(filtered_df)

# Reset stdout
sys.stdout = sys.__stdout__

# Get the printed output
printed_output = output.getvalue()

# Reset pandas display options to default
pd.reset_option('display.max_columns')
")

  # Print the captured output in R
  cat(py$printed_output)
}


In [11]:
if (to_python && !to_thai_all_years) {
    # Sys.setenv(PYTHONPATH = here("data-cleaning", "grouper"))

    # Step 6: Process each row of the DataFrame through `drg_seeker` and append results
    # if (!file.exists(here(checkpoint_8_path, paste0(checkpoint_8_prefix, "_", year_to_load, suffix, ".rds")))) {
    #   py_run_file(here("data-cleaning", "py_scripts", "run_drg_seeker.py"))
    #   output_dt <- as.data.table(py$output)
    #   saveRDS(output_dt, here(checkpoint_8_path, paste0(checkpoint_8_prefix, "_", year_to_load, suffix, ".rds")), compress = TRUE)
    #   cat(py$statements)
    # } else {
    #   output_dt <- readRDS(here(checkpoint_8_path, paste0(checkpoint_8_prefix, "_", year_to_load, suffix, ".rds")))
    # }
    # py_run_file(here("data-cleaning", "py_scripts", "run_drg_seeker.py"))
    py_run_string("
import pandas as pd
import numpy as np
import swifter
from grouper import seeker
import traceback
import sys
import io

# Initialize StringIO object to capture print statements
statements_io = io.StringIO()

# Redirect print statements to the StringIO object
sys.stdout = statements_io

# Initialize the necessary libraries
libs = seeker.Libraries()

# Define a function to instantiate a Patient object for each row
def process_patient(row, libs):
    try:
        # Convert the row to a dictionary and create a Patient object
        patient = seeker.Patient(row.to_dict(), libs)

        # Extract relevant attributes from the Patient object
        result = {
            'mdc': patient.mdc,
            'pdc': patient.pdc,
            # 'dc': patient.dc,
            'pccl': patient.pccl,
            'drg': patient.drg,
            'error_code': patient.error_code,
            'warning_code': patient.warning_code
        }

        return pd.Series(result)

    except Exception as e:
        # Log the error and row information for debugging
        print(f'''Error processing patient with id_series {row['id_series']}: {e}''')

        # Optionally, you can log more information such as row content or traceback
        traceback.print_exc()  # Print the full stack trace for more details

        # Return None or default values for the error case
        return pd.Series({
            'mdc': None,
            'pdc': None,
            # 'dc': None,
            'pccl': None,
            'drg': None,
            'error_code': None,
            'warning_code': None
        })

# Apply the Patient class directly to each row using swifter
pandas_df[['mdc', 'pdc',
        #    'dc',
           'pccl', 'drg', 'error_code', 'warning_code']] = pandas_df.swifter.apply(
    lambda row: process_patient(row, libs),
    axis=1
)

# Store the result in output to be retrieved by R
output = pandas_df.rename(columns={'drg': 'py_drg'})

# Define the desired column order
desired_columns = [
    'id_series', 'mdc', 'pdc',
    # 'dc',
    'pccl', 'py_drg', 'error_code',
    'warning_code'
]

# Reorder the DataFrame and drop any columns not in the desired list
output = output[desired_columns]

# # Define the renaming mapping
# rename_mapping = {
#     'patage': 'pat_age',
#     'patsex': 'pat_sex',
#     'birthweight': 'pat_bwt',
#     'discharge': 'clin_discharge',
#     'icd9_list': 'clin_rvs'
# }

# # Rename the columns
# output = output.rename(columns=rename_mapping)

# Capture the print statements
statements = statements_io.getvalue()

# Reset the stdout to default
sys.stdout = sys.__stdout__

# Return both the output and the captured print statements
output, statements
")
    output_dt <- as.data.table(py$output)
    saveRDS(output_dt, here(checkpoint_8_path, paste0(checkpoint_8_prefix, "_", year_to_load, suffix, ".rds")), compress = TRUE)
    cat(py$statements)
}


In [12]:
if (to_python && !to_thai_all_years) {
  # Assume 'pandas_df' is your DataFrame in Python after running 'run_drg_seeker.py'
  # And 'output' is the result from your Python script
  # Retrieve 'output' DataFrame from Python

  # Rename columns to match required names if necessary
  setnames(output_dt,
    old = c("drg", "pdc", "pccl", "error_code", "warning_code"),
    new = c("py_drg", "py_pdc", "py_pccl", "py_err", "py_warn"), skip_absent = TRUE
  )

  # Select only the required columns
  required_columns <- c("id_series", "py_drg", "py_pdc", "py_pccl", "py_err", "py_warn")
  output_dt <- output_dt[, ..required_columns]

  # # Define the format_id function
  # format_id <- function(x) {
  #   x <- as.character(x)
  #   integer_x <- suppressWarnings(as.integer(x))
  #   x <- trimws(formatC(integer_x, format = "f", digits = 0))
  #   x[x == "NA" | is.na(integer_x)] <- NA_character_
  #   x
  # }

  # # Apply format_id to 'id_series'
  # output_dt[, id_series := format_id(id_series)]

  # Adjust data types
  output_dt[, py_drg := as.character(py_drg)]
  output_dt[, py_pdc := as.character(py_pdc)]
  output_dt[, py_pccl := as.numeric(py_pccl)]

  # Convert 'py_err' and 'py_warn' to arrays (list of character vectors)
  array_columns <- c("py_err", "py_warn")

  process_error_warning_column <- function(col) {
    lapply(col, function(x) {
      # Flatten x to a character vector
      x <- unlist(x)
      x <- as.character(x)

      # If x is NULL or length zero after unlisting, return character(0)
      if (is.null(x) || length(x) == 0) {
        return(character(0))
      }

      # Remove any NA values from x
      x <- x[!is.na(x)]

      # Remove any "None", "NA", or empty strings from x
      x <- x[!(x %in% c("None", "NA", "NaN", ""))]

      # If x is now length zero after cleaning, return character(0)
      if (length(x) == 0) {
        return(character(0))
      }

      # Now split each element of x by comma and optional whitespace
      split_x <- unlist(strsplit(x, ",\\s*"))

      # Remove any empty strings, "NA", or "None" from split_x
      split_x <- split_x[!(split_x %in% c("", "NaN", "NA", "None")) & !is.na(split_x)]

      # Return character(0) if split_x is empty after cleaning
      if (length(split_x) == 0) {
        return(character(0))
      } else {
        return(split_x)
      }
    })
  }

  # Apply the processing function to the columns
  if (is_unix) {
    output_dt[, (array_columns) := mclapply(.SD, process_error_warning_column, mc.cores = parallel::detectCores()), .SDcols = array_columns]
  } else {
    output_dt[, (array_columns) := lapply(.SD, process_error_warning_column), .SDcols = array_columns]
  }

  # Now 'output_dt' is your final result
  # You can proceed to use 'output_dt' as needed

  # Replace <NA> values in 'py_drg' and 'py_pdc' with character(0)
  output_dt[, py_drg := ifelse(is.na(py_drg), "", py_drg)]
  output_dt[, py_pdc := ifelse(is.na(py_pdc), "", py_pdc)]
  # output_dt[, py_pccl := ifelse(is.nan(py_pccl), NA_real_, py_pccl)]

  # For example, print the first few rows
  # print(head(output_dt[id_series == 24465430]))
  print(head(output_dt, 100))
}


In [13]:
if (to_python && !to_thai_all_years) {
  if (to_debug) fwrite(output_dt, "test3.csv")
  # str(output_dt)
}


# BQ Upload


In [14]:
# Check for duplicates in id_series
if (to_python && !to_thai_all_years && to_bq && any(duplicated(output_dt$id_series))) {
  stop("The 'id_series' column contains duplicates. Execution stopped.")
}


In [15]:
if (to_python && !to_thai_all_years && to_bq) {
  # Set the table name based on row count
  bq_table <- if (nrow(output_dt) == total_rows) {
    paste0("python_", year_to_load)
  } else {
    paste0("temp_python_", year_to_load)
  }

  # Check if the table should be dropped and replaced
  tryCatch(
    {
      bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table))
      message("Table dropped successfully.\n")
    },
    error = function(e) {
      # If the table does not exist, just continue
      if (grepl("Not found", e, ignore.case = TRUE)) {
        message("Table does not exist, nothing to drop.\n")
      } else {
        # If it's a different error, re-throw the error
        stop(e)
      }
    }
  )

  # Attempt to create the table
  tryCatch(
    {
      bq_table_create(
        bq_table(gcp_proj, bq_dataset, bq_table),
        fields = fromJSON(here(
          "data-cleaning/r_scripts_v2",
          "bq_schema_thai.json"
        ), simplifyDataFrame = FALSE)
      )
      message("Table created successfully.\n")
    },
    error = function(e) {
      # Check if the error message indicates that the table already exists
      if (grepl("already exists", e, ignore.case = TRUE)) {
        message("Table already exists. Skipping creation and upload.")
      } else {
        # If it's a different error, re-throw the error
        stop(e)
      }
    }
  )

  # Upload to BQ only if table is empty
  if (to_write) {
    chunk_size <- 250000 # Adjust the chunk size based on memory availability
    num_chunks <- ceiling(nrow(output_dt) / chunk_size)

    for (i in seq_len(num_chunks)) {
      cat(paste("\rUploading chunk no.:", i))
      flush.console()
      chunk <- output_dt[
        ((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(output_dt)),
      ]

      bq_table_upload(
        bq_table(gcp_proj, bq_dataset, bq_table),
        values = chunk,
        write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
      )
      cat(paste("\rFinished uploading chunk no.:", i))
      flush.console()
    }
  }
}


# Thai

In [16]:
if (to_thai) {
  if (!to_thai_all_years) {
    if (to_generate_thai) {
      if (to_spc) {
        cat("\rReading stata\n")
        flush.console()
        result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata_subset_with_bdate", ".rds")))
      } else {
        cat("\rReading final\n")
        flush.console()
        result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_bdate_with_time", ".rds")))
      }
      # str(result)

      result_mapping <- result[, .(id_series, caseid)]
      cat("\rExporting for grouper\n")
      flush.console()
      # Define chunk size
      chunk_size <- 5000000
      num_chunks <- ceiling(nrow(result) / chunk_size)

      gcs_auth(email = gcs_email)

      for (i in seq_len(num_chunks)) {
        # Define the file path and name for this part
        output_file <- here(
          checkpoint_4_path,
          paste0(
            checkpoint_4_prefix, year_to_load, suffix,
            "part_", i, "_of_", num_chunks, ".txt"
          )
        )

        # Extract the chunk
        start_row <- (i - 1) * chunk_size + 1
        end_row <- min(i * chunk_size, nrow(result))
        chunk <- result[start_row:end_row, ]

        # Export the chunk to a file
        export_for_grouper(chunk, output_file, i)
        message("Saved part ", i, " of ", num_chunks, " to ", output_file)

        # Upload the file to GCS
        message("Uploading part ", i, " of ", num_chunks, " to GCS")
        gcs_upload(
          file = output_file,
          bucket = gcs_bucket,
          name = paste0(gcs_pre_fpath, "/", basename(output_file)),
          predefinedAcl = "bucketLevel"
        )

        # Clean up memory
        rm(chunk)
        gc()
      }
    } else {
      message("Skipping thai txt generation")
    }
  } else if (to_thai_all_years) {
    if (to_generate_thai) {
      for (year_to_load in c(2018:2023)) {
        year_to_load <<- year_to_load
        year_to_load <- year_to_load
        if (to_spc) {
          cat("\rReading stata\n")
          flush.console()
          result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata_subset_with_bdate", ".rds")))
        } else {
          cat("\rReading final\n")
          flush.console()
          result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_bdate_with_time", ".rds")))
        }
        # str(result)
        result_mapping <- result[, .(id_series, caseid)]
        cat("\rExporting for grouper\n")
        flush.console()
        # Define chunk size
        chunk_size <- 5000000
        num_chunks <- ceiling(nrow(result) / chunk_size)

        gcs_auth(email = gcs_email)

        for (i in seq_len(num_chunks)) {
          # Define the file path and name for this part
          output_file <- here(
            checkpoint_4_path,
            paste0(
              checkpoint_4_prefix, year_to_load, suffix,
              "part_", i, "_of_", num_chunks, ".txt"
            )
          )

          # Extract the chunk
          start_row <- (i - 1) * chunk_size + 1
          end_row <- min(i * chunk_size, nrow(result))
          chunk <- result[start_row:end_row, ]

          # Export the chunk to a file
          export_for_grouper(chunk, output_file, i)
          message("Saved part ", i, " of ", num_chunks, " to ", output_file)

          # Upload the file to GCS
          message("Uploading part ", i, " of ", num_chunks, " to GCS")
          gcs_upload(
            file = output_file,
            bucket = gcs_bucket,
            name = paste0(gcs_pre_fpath, "/", basename(output_file)),
            predefinedAcl = "bucketLevel"
          )

          # Clean up memory
          rm(chunk)
          gc()
        }
      }
    } else {
      message("Skipping thai txt generation")
    }
  }

  # Prompt for manual confirmation if needed
  if ((thai_prompt || to_prompt) && to_generate_thai && !to_thai_all_years) {
    response <- tolower(readline(prompt = "Have you run the Thai grouper manually? (y/n): "))
    if (response != "y") {
      stop("Thai Grouper not run yet. Script terminated. Continue on manually if necessary")
    }
    message("Continuing with the script...\n")
  } else {
    message("Thai Grouper is assumed to have been run already. Continuing with the script...\n")
  }

  if (!to_thai_all_years) {
    cat("\rDownloading Grouper results\n")
    flush.console()
    # Define chunk size
    if (to_spc) {
      cat("\rReading stata\n")
      flush.console()
      result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata_subset_with_bdate", ".rds")))
    } else {
      cat("\rReading final\n")
      flush.console()
      result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final_subset_with_bdate_with_time", ".rds")))
    }
    # str(result)
    result_mapping <- result[, .(id_series, caseid)]
    # Define chunk size
    chunk_size <- 5000000
    num_chunks <- ceiling(nrow(result) / chunk_size)

    gcs_auth(email = gcs_email)
    # Download each part and combine them into thai_result
    thai_result <- list()
    for (i in seq_len(num_chunks)) {
      # Define the remote file name and local path for this part
      remote_file <- paste0(
        gcs_post_fpath,
        "/",
        toupper(
          paste0(
            checkpoint_5_prefix, year_to_load, suffix,
            "part_", i, "_of_", num_chunks
          )
        ),
        "Res.TXT"
      )
      local_file <- here(
        checkpoint_5_path,
        paste0(
          toupper(
            paste0(
              checkpoint_5_prefix, year_to_load, suffix,
              "part_", i, "_of_", num_chunks
            )
          ),
          "Res.TXT"
        )
      )

      # Download the part from GCS
      message("Downloading part ", i, " of ", num_chunks, " from GCS")
      gcs_get_object(
        object_name = remote_file,
        bucket = gcs_bucket,
        saveToDisk = local_file,
        overwrite = TRUE
      )

      # Read the downloaded part and store it in the list
      part_data <- fread(local_file, colClasses = "character")
      thai_result[[i]] <- part_data

      # Clean up memory
      rm(part_data)
      gc()
    }

    # Combine all parts into a single data.table
    thai_result <- rbindlist(thai_result, use.names = FALSE, fill = FALSE)
    cat(paste("nrow thai_result:", nrow(thai_result), "\n"))
    cat(paste("nrow result_mapping:", nrow(result_mapping), "\n"))
    cat(paste("nrow result:", nrow(result), "\n"))
    # Final message
    message("All parts downloaded and combined successfully.\n")
    if (to_debug) print(head(thai_result))
    # Check for duplicates in id_series
    if (any(duplicated(thai_result$caseid))) {
      # Identify duplicates
      duplicate_ids <- thai_result$caseid[duplicated(thai_result$caseid)]

      # Extract rows with duplicate id_series
      duplicate_rows <- thai_result[caseid %in% duplicate_ids, ]

      # Print rows with duplicates
      cat("Rows with duplicate 'id_series':\n")
      print(duplicate_rows)

      # Stop execution
      stop("The 'id_series' column contains duplicates. Execution stopped.")
    }

    thai_result <- merge(
      thai_result,
      result_mapping, # Select only caseid and id_series from result_mapping
      by = "caseid", # Column to join on
      all.x = TRUE,
      all.y = FALSE,
    )
    cat(paste("nrow thai_result:", nrow(thai_result), "\n"))
    cat(paste("nrow result_mapping:", nrow(result_mapping), "\n"))
    cat(paste("nrow result:", nrow(result), "\n"))
    cat("\rRenaming columns\n")
    flush.console()
    thai_result[, caseid := id_series]
    thai_result[, id_series := NULL]
    thai_result[, thai_drg := drg]
    thai_result[, thai_rw := rw]
    thai_result[, thai_wtlos := wtlos]
    thai_result[, thai_ot := ot]
    thai_result[, thai_adjrw := adjrw]
    thai_result[, thai_err := err]
    thai_result[, thai_warn := warn]
    thai_result[, thai_los := los]
    thai_result[, drg := NULL]
    thai_result[, drgname := NULL]
    thai_result[, rw := NULL]
    thai_result[, wtlos := NULL]
    thai_result[, ot := NULL]
    thai_result[, adjrw := NULL]
    thai_result[, err := NULL]
    thai_result[, warn := NULL]
    thai_result[, los := NULL]

    # str(thai_result)
  }
}


Reading final
Exporting for grouper


ℹ 2024-11-18 12:52:34.219714 > Setting client.id from options(googleAuthR.client_id)



Classes ‘data.table’ and 'data.frame':	5000000 obs. of  42 variables:
 $ CASEID : chr  "1" "2" "3" "4" ...
 $ DOB    : chr  "03/02/1989" "06/09/1971" "03/02/2016" "21/09/1963" ...
 $ Sex    : num  2 2 1 2 2 2 2 1 2 2 ...
 $ DateAdm: chr  "03/02/2018" "06/09/2018" "02/02/2018" "21/09/2018" ...
 $ TimeAdm: chr  "2015" "1000" "0920" "1518" ...
 $ DateDsc: chr  "08/02/2018" "06/09/2018" "07/02/2018" "22/09/2016" ...
 $ TimeDsc: chr  "1400" "1100" "1345" "1711" ...
 $ DischT : chr  "1" "1" "1" "1" ...
 $ AdmWt  : chr  "--" "--" "--" "--" ...
 $ PDx    : chr  "O820" "--" "J189" "G629" ...
 $ SDx1   : chr  "Z370" "--" "B059" "--" ...
 $ SDx2   : chr  "--" "--" "--" "--" ...
 $ SDx3   : chr  "--" "--" "--" "--" ...
 $ SDx4   : chr  "--" "--" "--" "--" ...
 $ SDx5   : chr  "--" "--" "--" "--" ...
 $ SDx6   : chr  "--" "--" "--" "--" ...
 $ SDx7   : chr  "--" "--" "--" "--" ...
 $ SDx8   : chr  "--" "--" "--" "--" ...
 $ SDx9   : chr  "--" "--" "--" "--" ...
 $ SDx10  : chr  "--" "--" "--" "--" 

Saved part 1 of 2 to /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_4_thai_master_input/checkpoint_4_thai_grouper_input_2018_full_part_1_of_2.txt

Uploading part 1 of 2 to GCS

ℹ 2024-11-18 12:56:27.689773 > File size detected as  770.3 Mb

ℹ 2024-11-18 12:56:27.830456 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/phic-claims-checkpoints/o/?uploadType=resumable&name=pre-tdrg%2Fcheckpoint_4_thai_grouper_input_2018_full_part_1_of_2.txt&upload_id=AFiumC5nZT70BUBqFyfKXsW5oBhd9o1Tzi0pOB5uyGrdpEcC8SoqGqyOfJ6HTYeAlOno-Kw5EzRXfcjslgwRxsBDuftr11a0e4YqCgt1EXjeOT-4



Classes ‘data.table’ and 'data.frame':	3338176 obs. of  42 variables:
 $ CASEID : chr  "5000001" "5000002" "5000003" "5000004" ...
 $ DOB    : chr  "22/09/1950" "20/09/2012" "18/08/1963" "08/09/1948" ...
 $ Sex    : num  2 2 2 2 1 1 2 2 1 2 ...
 $ DateAdm: chr  "22/09/2018" "21/09/2018" "18/08/2018" "09/09/2018" ...
 $ TimeAdm: chr  "1100" "0935" "1400" "2145" ...
 $ DateDsc: chr  "24/09/2018" "24/09/2018" "20/08/2018" "13/09/2018" ...
 $ TimeDsc: chr  "1130" "1610" "1105" "1350" ...
 $ DischT : chr  "1" "1" "1" "1" ...
 $ AdmWt  : chr  "--" "--" "--" "--" ...
 $ PDx    : chr  "K291" "J450" "J189" "J450" ...
 $ SDx1   : chr  "E049" "--" "E789" "N390" ...
 $ SDx2   : chr  "--" "--" "I10" "--" ...
 $ SDx3   : chr  "--" "--" "--" "--" ...
 $ SDx4   : chr  "--" "--" "--" "--" ...
 $ SDx5   : chr  "--" "--" "--" "--" ...
 $ SDx6   : chr  "--" "--" "--" "--" ...
 $ SDx7   : chr  "--" "--" "--" "--" ...
 $ SDx8   : chr  "--" "--" "--" "--" ...
 $ SDx9   : chr  "--" "--" "--" "--" ...
 $ SDx10

...

Saved part 2 of 2 to /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_4_thai_master_input/checkpoint_4_thai_grouper_input_2022_full_part_2_of_2.txt

Uploading part 2 of 2 to GCS

ℹ 2024-11-18 13:15:53.610121 > File size detected as  121.2 Mb

ℹ 2024-11-18 13:15:53.683057 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/phic-claims-checkpoints/o/?uploadType=resumable&name=pre-tdrg%2Fcheckpoint_4_thai_grouper_input_2022_full_part_2_of_2.txt&upload_id=AFiumC59JywXTaH049nuApX4jUU8IEGZKt57DvoOOQw_cXHJQbgvSxpSCuCU20ghr3C-_Qa5uvuBabH5DpXJ2OHR5wj1VShIIel1MHPbRc5WXxOtWg



Reading final
Exporting for grouper
Classes ‘data.table’ and 'data.frame':	5000000 obs. of  42 variables:
 $ CASEID : chr  "1" "2" "3" "4" ...
 $ DOB    : chr  "01/01/1990" "01/01/2020" "04/01/1996" "05/01/1971" ...
 $ Sex    : num  2 1 2 1 1 2 1 2 2 1 ...
 $ DateAdm: chr  "01/01/2023" "01/01/2023" "04/01/2023" "05/01/2023" ...
 $ TimeAdm: chr  "2020" "1713" "1947" "1350" ...
 $ DateDsc: chr  "03/01/2023" "04/01/2023" "07/01/2023" "05/01/2023" ...
 $ TimeDsc: chr  "0820" "1632" "1042" "1430" ...
 $ DischT : chr  "1" "1" "1" "1" ...
 $ AdmWt  : chr  "--" "--" "--" "--" ...
 $ PDx    : chr  "O809" "K291" "I639" "--" ...
 $ SDx1   : chr  "Z370" "E86" "--" "--" ...
 $ SDx2   : chr  "Z392" "--" "--" "--" ...
 $ SDx3   : chr  "--" "--" "--" "--" ...
 $ SDx4   : chr  "--" "--" "--" "--" ...
 $ SDx5   : chr  "--" "--" "--" "--" ...
 $ SDx6   : chr  "--" "--" "--" "--" ...
 $ SDx7   : chr  "--" "--" "--" "--" ...
 $ SDx8   : chr  "--" "--" "--" "--" ...
 $ SDx9   : chr  "--" "--" "--" "--" ...


Saved part 1 of 2 to /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_4_thai_master_input/checkpoint_4_thai_grouper_input_2023_full_part_1_of_2.txt

Uploading part 1 of 2 to GCS

ℹ 2024-11-18 13:20:03.10634 > File size detected as  771.2 Mb

ℹ 2024-11-18 13:20:03.241109 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/phic-claims-checkpoints/o/?uploadType=resumable&name=pre-tdrg%2Fcheckpoint_4_thai_grouper_input_2023_full_part_1_of_2.txt&upload_id=AFiumC7s7Y_zkzYhvNelZxH2DAJcguV5xtRbCYcadXOWSJczpob3N-yfFhSmyLb_3vg9B2YJksstFd1D7AmCR2YenQOCUsnsYcq96l1dX4u5E7nexQ



Classes ‘data.table’ and 'data.frame':	1790706 obs. of  42 variables:
 $ CASEID : chr  "5000001" "5000002" "5000003" "5000004" ...
 $ DOB    : chr  "23/02/1982" "25/02/1996" "14/02/1954" "22/03/1991" ...
 $ Sex    : num  2 2 1 2 2 2 2 2 2 2 ...
 $ DateAdm: chr  "23/02/2023" "25/02/2023" "14/02/2023" "22/03/2023" ...
 $ TimeAdm: chr  "0110" "1125" "2325" "1430" ...
 $ DateDsc: chr  "25/02/2023" "28/02/2023" "16/02/2023" "24/03/2023" ...
 $ TimeDsc: chr  "1415" "1500" "1427" "1520" ...
 $ DischT : chr  "1" "1" "1" "1" ...
 $ AdmWt  : chr  "--" "--" "--" "--" ...
 $ PDx    : chr  "O809" "O619" "D649" "O820" ...
 $ SDx1   : chr  "Z370" "O820" "--" "O342" ...
 $ SDx2   : chr  "--" "Z370" "--" "Z370" ...
 $ SDx3   : chr  "--" "--" "--" "--" ...
 $ SDx4   : chr  "--" "--" "--" "--" ...
 $ SDx5   : chr  "--" "--" "--" "--" ...
 $ SDx6   : chr  "--" "--" "--" "--" ...
 $ SDx7   : chr  "--" "--" "--" "--" ...
 $ SDx8   : chr  "--" "--" "--" "--" ...
 $ SDx9   : chr  "--" "--" "--" "--" ...
 $ SD

Saved part 2 of 2 to /home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_4_thai_master_input/checkpoint_4_thai_grouper_input_2023_full_part_2_of_2.txt

Uploading part 2 of 2 to GCS

ℹ 2024-11-18 13:21:03.691315 > File size detected as  276.6 Mb

ℹ 2024-11-18 13:21:03.768214 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/phic-claims-checkpoints/o/?uploadType=resumable&name=pre-tdrg%2Fcheckpoint_4_thai_grouper_input_2023_full_part_2_of_2.txt&upload_id=AFiumC7d4UP6JQAnPredUsbajHiRa5rgqjmDxsEv1esjdtsU1f4Yt9st6KX1LtnSOjpbV9s1HEbv6dWPJs1dXQYtu3mPMxdTvG694EqgjaICi78c

Thai Grouper is assumed to have been run already. Continuing with the script...




In [17]:
if (!to_thai_all_years) {
  print(nrow(thai_result))
  print(nrow(thai_result[thai_err == "6"]))
}


In [18]:
if (to_python && to_thai && !to_thai_all_years) {
  if (exists("output_dt")) {
    before_merge <- data.table::copy(output_dt)
    # str(before_merge)
    before_merge[, caseid := id_series]

    if (to_debug) print(head(before_merge))
    merged <- merge(before_merge, thai_result, by = "caseid", all.x = TRUE)
    if (to_debug) print(head(merged))
    diff_merged <- merged[!as.character(ifelse(is.na(py_drg), "NA", py_drg)) == as.character(thai_drg)]
    print(nrow(diff_merged))
    fwrite(diff_merged, here(checkpoint_9_path, paste0("checkpoint_9_grouper_differences_", year_to_load, suffix, ".csv")))
  }
}


In [19]:
if (to_python && to_thai && !to_thai_all_years) {
  if (exists("merged")) { # str(merged)
    if (to_debug) fwrite(merged, "test4.csv")
  }
}


In [20]:
if (to_thai && !to_thai_all_years) {
  # Please run thai grouper first
  result_after_thai <- data.table::copy(thai_result)

  # Convert data types to match BigQuery schema
  # format as full numbers, no exponential form
  result_after_thai[, id_series := caseid]
  result_after_thai[, caseid := NULL]
  # format as full numbers, no exponential form
  # result_after_thai[, id_pin := trimws(formatC(as.integer(id_pin), format = "f", digits = 0))]
  # result_after_thai[, date_adm := as.Date(date_adm, format = "%Y-%m-%d")]
  # result_after_thai[, time_adm := as.ITime(time_adm)]
  # result_after_thai[, date_dis := as.Date(date_dis, format = "%Y-%m-%d")]
  # result_after_thai[, time_dis := as.ITime(time_dis)]
  # result_after_thai[, date_rec := as.Date(date_rec, format = "%Y-%m-%d")]
  # result_after_thai[, date_ref := as.Date(date_ref, format = "%Y-%m-%d")]
  # result_after_thai[, date_check := as.Date(date_check, format = "%Y-%m-%d")]
  # result_after_thai[, id_hci := as.character(id_hci)]
  # result_after_thai[, id_hcp := id_hcp] # as is

  # # Convert character "0"/"1" to logical for Boolean fields
  # result_after_thai[, clin_outpatient := as.logical(clin_outpatient)] # as is
  # result_after_thai[, clin_emergency := as.logical(clin_emergency)] # as is

  # result_after_thai[, pat_type := as.character(pat_type)]
  # result_after_thai[, clin_acc := as.character(clin_acc)]
  # result_after_thai[, pat_rel := as.character(pat_rel)]
  # result_after_thai[, pat_bdate := as.Date(pat_bdate, format = "%Y-%m-%d")]
  # result_after_thai[, pat_age := as.numeric(pat_age)]
  # result_after_thai[, pat_sex := as.character(pat_sex)]
  # result_after_thai[, pat_bwt := as.numeric(pat_bwt)]
  # result_after_thai[, pat_memcat_parent := as.character(pat_memcat_parent)]
  # result_after_thai[, pat_memcat_child := as.character(pat_memcat_child)]
  # result_after_thai[, clin_discharge := as.integer(clin_discharge)]
  # # result[, clin_c1 := clin_c1]
  # # result[, clin_c2 := clin_c2]

  # result_after_thai[, claim_status := as.character(claim_status)]
  # result_after_thai[, claim_payout := as.numeric(claim_payout)]
  # result_after_thai[, claim_charge := as.numeric(claim_charge)]
  # result_after_thai[, date_ext := as.Date(date_ext, format = "%Y-%m-%d")]
  # result_after_thai[, id_year := as.integer(id_year)]

  # if (is_unix) {
  #   result_after_thai[, clin_sdx := mclapply(clin_sdx, function(x) if (all(is.na(x))) character(0) else x)]
  # } else {
  #   result_after_thai[, clin_sdx := future_lapply(clin_sdx, function(x) if (all(is.na(x))) character(0) else x)]
  # }

  # result_after_thai[, clin_proc := clin_rvs] # as is
  # if (is_unix) {
  #   result_after_thai[, clin_proc := mclapply(clin_proc, function(x) if (all(is.na(x))) character(0) else x)]
  # } else {
  #   result_after_thai[, clin_proc := future_lapply(clin_proc, function(x) if (all(is.na(x))) character(0) else x)]
  # }

  # result_after_thai[, clin_rvs := NULL] # as is
  # result_after_thai[, pat_ageday := as.integer(ageday)] # as is
  # result_after_thai[, ageday := NULL]

  # result_after_thai[, clin_pdx := as.character(clin_pdx)]
  # result_after_thai[, clin_pdx_source := as.integer(pdx_code)]
  # result_after_thai[, pdx_code := NULL]

  result_after_thai[, thai_drg := as.character(thai_drg)]
  result_after_thai[, thai_rw := as.numeric(thai_rw)]
  result_after_thai[, thai_wtlos := as.numeric(thai_wtlos)]
  result_after_thai[, thai_ot := as.integer(thai_ot)]
  result_after_thai[, thai_adjrw := as.numeric(thai_adjrw)]
  result_after_thai[, thai_err := as.integer(thai_err)]
  result_after_thai[, thai_warn := as.integer(thai_warn)]
  result_after_thai[, thai_los := as.integer(thai_los)]

  # result_after_thai[, py_pdc := as.character(py_pdc)]
  # result_after_thai[, py_pccl := as.numeric(py_pccl)]
  # result_after_thai[, py_drg := as.character(py_drg)]
  # result_after_thai[, py_warn := py_warn] # as is
  # result_after_thai[, py_err := py_err] # as is

  # Reorder the columns in the result data.table to match the schema
  setcolorder(result_after_thai, c(
    # "caseid",
    # "id_year",
    "id_series",
    # "id_pin",
    # "id_hci",
    # "id_hcp",
    # "date_adm",
    # "time_adm",
    # "date_dis",
    # "time_dis",
    # "date_rec",
    # "date_ref",
    # "date_check",
    # "date_ext",
    # "pat_type",
    # "pat_rel",
    # "pat_bdate",
    # "pat_age",
    # "pat_ageday",
    # "pat_sex",
    # "pat_bwt",
    # "pat_memcat_parent",
    # "pat_memcat_child",
    # "claim_status",
    # "claim_payout",
    # "claim_charge",
    # "clin_discharge",
    # "clin_outpatient",
    # "clin_emergency",
    # "clin_acc",
    # "clin_c1",
    # "clin_c2",
    # "clin_sdx",
    # "clin_proc",
    # "clin_pdx",
    # "clin_pdx_source",
    "thai_drg",
    "thai_rw",
    "thai_wtlos",
    "thai_ot",
    "thai_adjrw",
    "thai_err",
    "thai_warn",
    "thai_los" # ,
    # "py_drg",
    # "py_pdc",
    # "py_pccl",
    # "py_warn",
    # "py_err"
  ))

  # result_after_thai[, icd9_list := NULL]
  # result_after_thai[, pat_age_orig := NULL]

  # result_after_thai[, c1_orig := NULL]
  # result_after_thai[, c2_orig := NULL]
  # result_after_thai[, c1 := NULL]
  # result_after_thai[, c2 := NULL]
}


In [21]:
if (to_thai && !to_thai_all_years) {
  # str(result_after_thai)
  print(result_after_thai[grepl("e", id_series)])
  # print(result_after_thai[grepl("e", id_pin)])
  # print(result_after_thai[grepl("e", id_hci)])
}


In [22]:
if (to_thai && !to_thai_all_years) {
  # str(result_after_thai)
}


In [23]:
if (to_thai && !to_thai_all_years) {
  # Check for duplicates in id_series
  if (any(duplicated(result_after_thai$id_series))) {
    # Identify duplicates
    duplicate_ids <- result_after_thai$id_series[duplicated(result_after_thai$id_series)]

    # Extract rows with duplicate id_series
    duplicate_rows <- result_after_thai[id_series %in% duplicate_ids, ]

    # Print rows with duplicates
    cat("Rows with duplicate 'id_series':\n")
    print(duplicate_rows)

    # Stop execution
    stop("The 'id_series' column contains duplicates. Execution stopped.")
  }
}


In [ ]:
print(nrow(result))
print(nrow(result_after_thai))


[1] 6790706


ERROR: Error: object 'result_after_thai' not found


In [ ]:
print(result_after_thai[is.na(thai_drg)])
print(result_after_thai[is.na(id_series)])


In [ ]:
if (to_thai && !to_thai_all_years && to_bq) {
  saveRDS(result_after_thai, here(checkpoint_6_path, paste0(checkpoint_6_prefix, year_to_load, suffix, ".rds")))

  # message("You may now run drg-spc-v2.ipynb in the background")

  # Set the table name based on row count and sample status
  prefix <- if (nrow(result_after_thai) == nrow(result) && !to_sample) "thai_" else "temp_thai_"
  bq_table <- paste0(prefix, year_to_load)

  # Only proceed with BigQuery upload if `to_spc` is FALSE
  if (!to_spc) {
    # Check if the table should be dropped and replaced
    tryCatch(
      {
        bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table))
        message("Table dropped successfully.\n")
      },
      error = function(e) {
        # If the table does not exist, just continue
        if (grepl("Not found", e, ignore.case = TRUE)) {
          message("Table does not exist, nothing to drop.\n")
        } else {
          # If it's a different error, re-throw the error
          stop(e)
        }
      }
    )

    # Attempt to create the table
    tryCatch(
      {
        bq_table_create(
          bq_table(gcp_proj, bq_dataset, bq_table),
          fields = fromJSON(here(
            "data-cleaning/r_scripts_v2",
            "bq_schema_thai.json"
          ), simplifyDataFrame = FALSE)
        )
        message("Table created successfully.\n")
      },
      error = function(e) {
        # Check if the error message indicates that the table already exists
        if (grepl("already exists", e, ignore.case = TRUE)) {
          message("Table already exists. Skipping creation and upload.")
        } else {
          # If it's a different error, re-throw the error
          stop(e)
        }
      }
    )

    # Upload to BQ only if table is empty
    if (to_write) {
      chunk_size <- 250000 # Adjust the chunk size based on memory availability
      num_chunks <- ceiling(nrow(result_after_thai) / chunk_size)

      for (i in seq_len(num_chunks)) {
        cat(paste("\rUploading chunk no.:", i))
        flush.console()
        chunk <- result_after_thai[
          ((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result_after_thai)),
        ]

        bq_table_upload(
          bq_table(gcp_proj, bq_dataset, bq_table),
          values = chunk,
          write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
        )
        cat(paste("\rUploaded chunk no.:", i))
        flush.console()
      }
    }
  } else {
    message("Skipping BigQuery upload as to_spc is TRUE.")
  }
}
